# ML Laboratuvarı · Modül 2, Ünite 9: Bir öngörüyü dürüstçe değerlendirmek

**Prof. Dr. Mehmet Songur** · ML Laboratuvarı, Modül 2

Veri: ABD işsizlik oranı (FRED: `UNRATE`, aylık, %). Tablo: `issizlik` (sütunlar: `tarih`, `oran`). Kodu **Gemini'ye yazdır**, sonra **denetle**.

**Kurallar**
1. **Kurulum** ve **Kontrol** hücrelerini değiştirme; yalnızca çalıştır.
2. Görev hücrelerinde kodu Gemini ile üret; çalıştırmadan önce oku.
3. Kontrol hücresi geçerse bir **tamamlama kodu** verir; onu ML Laboratuvarı'ndaki kutuya yapıştır.

> Not: 2025 Ekim'in değeri eksik (ABD federal hükümetinin kapanması sırasında anket derlenemedi). Doldurma; nasıl ele alındığını denetle.

In [ ]:
#@title Kurulum — çalıştır, değiştirme

import io, base64, urllib.request
import numpy as np
import pandas as pd

def fred_indir(seri_kodu):
    """FRED'den bir seriyi indirir; sütunlar: tarih, <seri_kodu>. FRED eksik değerleri '.' ile gösterir;
    bunlar bilerek NaN yapılır ve sayısı yazdırılır."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + seri_kodu
    with urllib.request.urlopen(url) as yanit:
        df = pd.read_csv(io.BytesIO(yanit.read()))
    df.columns = ["tarih", seri_kodu]
    df["tarih"] = pd.to_datetime(df["tarih"])
    df[seri_kodu] = pd.to_numeric(df[seri_kodu], errors="coerce")
    print(f"{seri_kodu}: {len(df)} satır, {df[seri_kodu].isna().sum()} eksik değer, {df['tarih'].min():%Y-%m} – {df['tarih'].max():%Y-%m}")
    return df

issizlik = fred_indir("UNRATE").rename(columns={"UNRATE": "oran"}).sort_values("tarih").reset_index(drop=True)
_kopya = issizlik.copy()
KESIM = pd.Timestamp("2015-01-01")

def _k(s):
    return base64.b64decode(s).decode()

def _beklenen():
    d = _kopya.copy()
    # Takvim korunur: eksik ay satırı silinmeden gecikme oluşturulur
    d["onceki"] = d["oran"].shift(1)
    gecerli = d["oran"].notna() & d["onceki"].notna()
    egt = d[(d["tarih"] < KESIM) & gecerli]
    tst = d[(d["tarih"] >= KESIM) & gecerli]
    saf = float(np.sqrt(((tst["oran"] - tst["onceki"]) ** 2).mean()))
    b, a = np.polyfit(egt["onceki"], egt["oran"], 1)
    model = float(np.sqrt(((tst["oran"] - (a + b * tst["onceki"])) ** 2).mean()))
    hepsi = d[gecerli]
    b_hepsi, a_hepsi = np.polyfit(hepsi["onceki"], hepsi["oran"], 1)
    disari = d[(d["tarih"] >= KESIM) & ~gecerli]
    return dict(saf=saf, a=a, b=b, model=model, a_hepsi=a_hepsi, b_hepsi=b_hepsi, egt=egt, tst=tst, disari=disari)

def _sonlu(x):
    try:
        return np.isfinite(float(x))
    except Exception:
        return False

def _takvim_kontrol():
    t = _kopya["tarih"]
    if not t.is_monotonic_increasing or t.duplicated().any():
        return "kaynak tarih dizisi artan ve tekil değil"
    aylar = pd.period_range(t.min(), t.max(), freq="M")
    if not pd.PeriodIndex(t.dt.to_period("M")).equals(aylar):
        return "kaynak takvimde atlanan ay var"
    return None

def _gecikme_kontrol(tablo, ad, beklenen):
    if "onceki" not in tablo.columns:
        return f"❌ '{ad}' tablosunda 'onceki' sütunu yok (takvimde bir önceki ayın oranı)."
    if tablo["tarih"].duplicated().any():
        return f"❌ '{ad}' tablosunda bazı aylar birden çok kez geçiyor."
    s = tablo.set_index("tarih")
    ok = s["oran"].notna() & s["onceki"].notna()
    ogr = set(pd.to_datetime(s.index[ok]))
    bek = set(pd.to_datetime(beklenen["tarih"]))
    fazla = sorted(ogr - bek)
    if fazla:
        return f"❌ '{ad}' tablosunda gecikmesi geçerli olmaması gereken aylar var: {[f'{x:%Y-%m}' for x in fazla[:3]]}. Eksik ay satırı gecikmeden önce silindiyse bir sonraki ay yanlış aya bağlanır; önce takvimi koru, sonra shift(1) uygula."
    eksik = sorted(bek - ogr)
    if eksik:
        return f"❌ '{ad}' tablosunda hesaplanabilir {len(eksik)} ay eksik ya da gecikmesi boş (ilki {eksik[0]:%Y-%m})."
    v = s.loc[beklenen["tarih"], "onceki"].to_numpy(dtype=float)
    fark = np.abs(v - beklenen["onceki"].to_numpy(dtype=float)) > 1e-9
    if fark.any():
        ay = pd.Timestamp(beklenen["tarih"].iloc[int(np.argmax(fark))])
        return f"❌ '{ad}' tablosunda {ay:%Y-%m} için 'onceki' değeri takvimde bir önceki ayın oranı değil. Gecikme yanlış aya bağlanmış."
    return None

def kontrol_A(egitim, test, kok_hko_saf):
    for ad, t in [("egitim", egitim), ("test", test)]:
        if "tarih" not in t.columns or "oran" not in t.columns:
            print(f"❌ '{ad}' tablosunda 'tarih' ve 'oran' sütunları olmalı."); return
    sorun = _takvim_kontrol()
    if sorun:
        print("❌ Beklenmeyen kaynak durumu: " + sorun); return
    if pd.Timestamp(egitim["tarih"].max()) >= KESIM or pd.Timestamp(test["tarih"].min()) < KESIM:
        print("❌ Bölme zamana göre değil: eğitim 2015 öncesi, test 2015 ve sonrası olmalı. Veri karıştırılmış olabilir."); return
    e = _beklenen()
    for ad, t, bek in [("egitim", egitim, e["egt"]), ("test", test, e["tst"])]:
        sorun = _gecikme_kontrol(t, ad, bek)
        if sorun:
            print(sorun); return
    if not _sonlu(kok_hko_saf):
        print("❌ 'kok_hko_saf' sonlu bir sayı değil (NaN ya da sonsuz)."); return
    if abs(float(kok_hko_saf) - e["saf"]) > 1e-4:
        print("❌ Saf öngörünün kök HKO'su beklenenden farklı. Öngörü 'onceki' mi? Hata kare alınıp ortalanıp karekökü alındı mı?"); return
    dis = [f"{x:%Y-%m}" for x in e["disari"]["tarih"]]
    print(f"Test: {len(e['tst'])} ay değerlendirildi. Hesaplanamayan aylar: {dis} (gerçekleşme ya da önceki ay değeri eksik).")
    print(f"✅ Tüm kontroller geçti (saf öngörü kök HKO ≈ {e['saf']:.4f}). Tamamlama kodun: " + _k("TUwyLVNBRi04UlRR"))

def kontrol_B(a, b, kok_hko_model):
    for ad, x in [("a", a), ("b", b), ("kok_hko_model", kok_hko_model)]:
        if not _sonlu(x):
            print(f"❌ '{ad}' sonlu bir sayı değil (NaN ya da sonsuz). Eksik değerler kestirime girmiş olabilir."); return
    e = _beklenen()
    egitim, test = globals().get("egitim"), globals().get("test")
    if abs(float(b) - e["b_hepsi"]) < 0.002 and abs(float(b) - e["b"]) > 0.002:
        print("❌ Katsayılar tüm veriyle (test dönemi dahil) kestirilmiş görünüyor: bu bir sızıntı. Yalnızca eğitim verisini kullan."); return
    if abs(float(b) - e["b"]) > 1e-6 or abs(float(a) - e["a"]) > 1e-6:
        print("❌ Katsayılar beklenenden farklı. Model 'oran_t = a + b · oran_t-1' mi? Yalnızca eğitimdeki geçerli aylarla mı kestirildi?"); return
    if test is None or "tahmin" not in test.columns:
        print("❌ 'test' tablosunda 'tahmin' sütunu yok (a + b · onceki)."); return
    s = test.set_index("tarih")
    t = s.loc[e["tst"]["tarih"], "tahmin"].to_numpy(dtype=float)
    if not np.allclose(t, float(a) + float(b) * e["tst"]["onceki"].to_numpy(dtype=float), atol=1e-9):
        print("❌ 'tahmin' sütunu bazı aylarda a + b · onceki değil. Satırlar kaymış ya da yanlış katsayılar kullanılmış olabilir."); return
    if abs(float(kok_hko_model) - e["model"]) > 1e-4:
        print("❌ Modelin test kök HKO'su beklenenden farklı. Test dönemindeki geçerli aylarda mı hesaplandı?"); return
    print(f"Eğitim: {len(e['egt'])} ay ({e['egt']['tarih'].min():%Y-%m} – {e['egt']['tarih'].max():%Y-%m}); test: {len(e['tst'])} ay.")
    print(f"Model ≈ {e['model']:.4f}, saf öngörü ≈ {e['saf']:.4f}; fark ≈ {e['saf'] - e['model']:.4f} yüzde puan. Bir ay ileri, koşullu değerlendirme: her ay önceki ayın gerçekleşmesi biliniyor varsayılıyor.")
    print("✅ Tüm kontroller geçti. Tamamlama kodun: " + _k("TUwyLU1ETC01SFdD"))

print("issizlik tablosu:", issizlik.shape)
issizlik.tail()

## Görev A — Zamana göre bölme ve kıyas ölçütü

1. Önce **takvimi doğrula**: tarihler artan ve tekil mi, hiçbir ay atlanmamış mı? Ekim 2025 satırı değeri boş olarak duruyor; **silme**.
2. Tam takvim üzerinde gecikmeyi oluştur: `issizlik["onceki"] = issizlik["oran"].shift(1)`. Gecikmeyi eksik satırları sildikten sonra kurarsan Kasım 2025'in “önceki ayı” Eylül olur.
3. Tabloyu **zamana göre** böl: `tarih < 2015-01-01` → `egitim`, `tarih >= 2015-01-01` → `test`. Karıştırma yok. İki tabloda da `onceki` sütunu kalsın.
4. Test döneminde `oran` ve `onceki` değerinin ikisi de bulunan aylarda **saf öngörünün** (bu ay = önceki ay) kök HKO'sunu hesapla ve `kok_hko_saf` adlı değişkene ata. Hesaplanamayan ayları ve değerlendirilen ay sayısını yazdır.

**Öngörü anı:** Her ay için öngörü, bir önceki ayın verisi **yayımlandıktan sonra** yapılıyor varsayılıyor (bir ay ileri, koşullu değerlendirme). Gerçekte ilk açıklanan rakamlar sonradan revize edilebilir; FRED'in bugünkü serisi revize edilmiş değerleri içerir.

**Kontrol A neyi sınar?** Bölmenin zamana göre olduğunu; iki tablodaki her geçerli ayın `onceki` değerinin takvimde bir önceki ayın oranı olduğunu (satır satır); geçerli olmaması gereken bir ayın (Kasım 2025) hesaba girmediğini; kök HKO'nun sonlu ve doğru olduğunu.

**Denetim soruları:** `train_test_split` ya da `sample` var mı? Kesim tarihi doğru mu? Gecikme eksik satır silinmeden önce mi kuruldu?

In [ ]:
# Görev A: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol A — çalıştır, değiştirme

In [ ]:
kontrol_A(egitim, test, kok_hko_saf)

## Görev B — Basit bir regresyon

`oran_t = a + b · oran_t-1` modelini **yalnızca `egitim` tablosunun geçerli aylarıyla** (`oran` ve `onceki` dolu) kestir. Katsayıları `a` ve `b` değişkenlerine ata. Test tablosuna `tahmin = a + b · onceki` sütununu ekle; test dönemindeki kök HKO'yu `kok_hko_model` değişkenine ata ve saf öngörüyle karşılaştır.

**Kıyasın kapsamı:** Bu, bir ay ileri koşullu bir değerlendirmedir: her test ayında bir önceki ayın gerçekleşmesi biliniyor sayılır. 2015 başında tek seferde bütün test dönemini öngörmek farklı (ve çok daha zor) bir sorudur. Fark küçükse, test dönemini ve veri sürümünü (indirme tarihi) birlikte raporla.

**Kontrol B neyi sınar?** Katsayıların sonlu olduğunu ve yalnızca eğitimdeki geçerli aylarla kestirildiğini; `tahmin` sütununun her geçerli test ayında a + b · onceki olduğunu; kök HKO'nun doğru olduğunu.

**Denetim soruları:** Katsayılar hangi veriyle kestirildi (`pd.concat` var mı)? Test döneminin ilk ayı için geçen ayın değeri nereden geliyor? Model saf öngörüyü ne kadar geçiyor?

In [ ]:
# Görev B: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol B — çalıştır, değiştirme

In [ ]:
kontrol_B(a, b, kok_hko_model)

## Denetim notların

- Gemini veriyi nasıl böldü? Değiştirmen gerekti mi?
- Regresyon katsayıları hangi veriyle kestirildi?
- Model saf öngörüyü geçiyor mu, ne kadar?